# Knowledge Distillation with hf_distiller
This notebook demonstrates:
1. Loading a teacher model from Hugging Face Hub
2. Creating a smaller student model
3. Preparing a toy dataset
4. Training the student using knowledge distillation
5. Visualizing training loss and logits comparison

You can replace the demo dataset with your own dataset for real training.

In [1]:
# Step 0 — Install requirements (run only once)
!pip install --no-deps git+https://github.com/Dhiraj309/transformers_distillation.git

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/Dhiraj309/transformers_distillation.git to c:\users\patil\appdata\local\temp\pip-req-build-_93suvg_
  Resolved https://github.com/Dhiraj309/transformers_distillation.git to commit eec0c657b772f842c30878f7d36fcf69731e3f21
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for transformers_distiller: filename=transformers_distiller-0.1.0-py3-none-any.whl size=11639 sha256=f4c101bf67e2ea8b6d103fd7e6ca93a3c90081ed55cec426a1711928811a6ea0
  Stored in directory: C:\Users\patil\AppData\Local\Temp\pip-ephem-wheel-cache-c7cjlwlf\wheels\0d\22\7a\7b6f72d21e3a6e4f60a7b03fda4acb5cfeeb146b6c0ea5c5e8
Succe

  Running command git clone --filter=blob:none --quiet https://github.com/Dhiraj309/transformers_distillation.git 'C:\Users\patil\AppData\Local\Temp\pip-req-build-_93suvg_'


## Step 1 — Imports and Setup

In [2]:
import sys
import os
from transformers import AutoTokenizer, TrainingArguments
from datasets import Dataset
from transformers_distillation.models import load_teacher, load_student
from transformers_distillation import DistillTrainer
import torch

c:\Users\patil\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 — Load Teacher Model

In [3]:
MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'

# Load teacher and tokenizer
teacher = load_teacher(model_name_or_path=MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Teacher model loaded:", teacher.__class__.__name__)
print("Tokenizer vocab size:", len(tokenizer))

Teacher model loaded: LlamaForCausalLM
Tokenizer vocab size: 49152


## Step 3 — Create Student Model
A smaller architecture for faster inference and lower memory usage.

In [4]:
student = load_student(
    model_name_or_path=MODEL_NAME,
    from_scratch=True,
    n_layers=4,
    n_heads=4,
    n_embd=256,
    is_pretrained=False
)
print("Student model created:", student.__class__.__name__)

Student model created: LlamaForCausalLM


## Step 4 — Prepare Dataset
Small in-memory dataset for demonstration. Replace with your own data for real training.

In [5]:
texts = [
    "Hello world!",
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming industries.",
    "Once upon a time, there was a curious developer.",
    "PyTorch makes deep learning both fun and powerful."
]
dataset = Dataset.from_dict({"text": texts})

def tokenize(batch):
    return tokenizer(batch['text'], max_length=128, padding=True, truncation=True)

tokenized_dataset = dataset.map(tokenize, remove_columns=['text'])
eval_dataset = tokenized_dataset.select(range(1))
print("Tokenized example:", tokenized_dataset[0])

Map: 100%|██████████| 5/5 [00:00<00:00, 92.21 examples/s]

Tokenized example: {'input_ids': [19556, 905, 17], 'attention_mask': [1, 1, 1]}


## Step 5 — Define Training Arguments

In [6]:
training_args = TrainingArguments(
    output_dir='./student-llm',
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=100,
    save_total_limit=5,
    report_to='none',
    lr_scheduler_type='cosine',
    warmup_steps=10,
)

## Step 6 — Initialize Distillation Trainer

In [7]:
trainer = DistillTrainer(
    teacher_model=teacher,
    student_model=student,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    training_args=training_args,
    kd_alpha=0.5,
    temperature=2.0,
    is_pretrained=False
)

c:\Users\patil\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers_distillation\trainer.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `DistillationTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


## Step 7 — Train Student Model
The student learns from both teacher outputs and ground truth labels.

In [8]:
# Keep track of loss for visualization
trainer_state = trainer.train()
losses = trainer_state.training_loss if hasattr(trainer_state, 'training_loss') else []

Step,Training Loss
1,27.726200
2,46.899300
3,73.354600
4,12.300200
5,49.185500
6,70.402400
7,47.146400
8,44.568800
9,25.926500
10,9.427700


## Step 8 — Evaluate Student Model

In [9]:
results = trainer.evaluate(eval_dataset = eval_dataset)
print('Evaluation results:', results)

Evaluation results: {'eval_runtime': 0.0305, 'eval_samples_per_second': 32.834, 'eval_steps_per_second': 32.834, 'epoch': 3.0}
